#### Setup

In [ ]:
import os, time
from pathlib import Path

from dvc_dat.ledger import Ledger, bump
from dvc_dat.ledger import LockTimeout, Unlocked

scratch_ledger_path = Path("scratch/triples-demo/ledger").resolve()

# Ledger: triples

The ledger as a quad store and nothing else. Every fact is a row
`(rev, subject, predicate, object)`, every write is one
transaction, every read is a function of one revision. No `run`
and no value functions here: the running example is plain
metadata about games (`game/G1`: teams, date, court, officials)
and a few clock subjects (`db/repo/demo-game/demo_game`, `schema/tape`).

## One constructor

In [ ]:
L = Ledger(scratch_ledger_path, create=True)   # made, then opened

`Ledger(location)` opens what is there and raises
`LedgerError` when nothing is; `create=True` is the one explicit
act that makes a ledger, so an empty one never appears by accident.

In [ ]:
L = Ledger(scratch_ledger_path)                # reopened by path

### A project's `.datconfig.yaml`

In real code a project names its ledger once and every script in
it says `Ledger()` with no argument. `Ledger(create=True)` with no
path makes a project here: it writes `.datconfig.yaml` with
`default: root` and creates the root ledger, a SQLite folder under
`data/` (a `root:` key would say where instead).

In [ ]:
project_path = Path("scratch/triples-demo/project").resolve()
project_path.mkdir(parents=True, exist_ok=True)
os.chdir(project_path)
L = Ledger(create=True)                   # .datconfig.yaml + ledger

print((project_path / ".datconfig.yaml").read_text())
# default: root
# namespace: ledger

Those lines are the whole config this notebook needs: the active
ledger is the root, and the project's names are its ledger's. The same file carries a
project's `mounts:` (the values and dats notebooks), and dvc-dat's
own keys; it is the file the dat manager already reads. `Ledger()`
walks up from the working directory to the nearest `.datconfig.yaml`
and opens what its `default:` names; besides the root, made on
first use when no `root:` names it, it never creates anything.

In [ ]:
L = Ledger()                              # the project's ledger

## One transaction is one revision

In [ ]:
with L.transaction(by="human:dan",
                   why="season opener") as t:
    t.add("game/G1", "home", "G4")
    t.add("game/G1", "away", "G5")
    t.add("game/G1", "date", "2026-10-11")
    t.add("game/G1", "court", "north")
rev1 = t.rev
print(rev1.id, rev1.tx)              # 1 db/tx/1
print(rev1.get(rev1.tx, "db/by"))    # human:dan
print(rev1.get(rev1.tx, "db/time"))  # 2026-10-18T15:52:07Z (UTC)
print(sorted(p for p, o in rev1.all(rev1.tx)))
# ['db/by', 'db/time', 'why']
print(rev1.get(rev1.tx, "why"))      # season opener

Every revision carries the wall-clock stamp `db/time` (UTC) beside
`by`. The stamp is a row like any other, so `history` and `since=`
work on it too.

`by` is required, once per transaction. Every other keyword is a
row on the transaction's own subject, `db/tx/<rev>`, and stays
bare: user vocabulary, so `why` has no prefix and `db/by` does.

### An exception in the body writes nothing

The `with` block is the transaction: a clean exit commits one
revision, an exception rolls back everything the body added.
Notice the top did not move and `court` is still `north`.

In [ ]:
try:
    with L.transaction(by="human:dan") as t:
        t.add("game/G1", "court", "east")
        raise RuntimeError("changed my mind")
except RuntimeError as e:
    print("RuntimeError:", e)
print(L.at().id == rev1.id)              # True: nothing written
print(L.at().get("game/G1", "court"))    # north

## A second transaction, and the ledger as rows

One more transaction: the court moves, and two officials are
named. `t[s, p] = o` is an `add` (more under The `[]` accessor).

In [ ]:
with L.transaction(by="human:dan") as t:
    t["game/G1", "court"] = "south"
    t.add("game/G1", "officials", "kim")
    t.add("game/G1", "officials", "ortiz")
rev2 = t.rev

Every row so far, unfolded, in revision order: `history()` is the
ledger as it is stored, the one read that does not fold. Notice
the two game transactions stacked, a second `court` row after the
first, and two `officials` rows side by side.

In [ ]:
rows = L.at().history()                   # every row, unfolded
print(f"{'rev':>3}  {'subject':<18} {'predicate':<15} object")
for r in rows:
    print(f"{r.rev.id:>3}  {r.subject:<18} {r.predicate:<15} "
          f"{r.object}")
# rev  subject            predicate       object
#   1  game/G1            home            G4
#   1  game/G1            away            G5
#   1  game/G1            date            2026-10-11
#   1  game/G1            court           north
#   1  db                 db/format       dvc-dat-ledger/1
#   1  db/tx/1            db/by           human:dan
#   1  db/tx/1            db/time         2026-10-18T15:52:07Z
#   1  db/tx/1            why             season opener
#   2  game/G1            court           south
#   2  game/G1            officials       kim
#   2  game/G1            officials       ortiz
#   2  db/tx/2            db/by           human:dan

## Folding semantics

Every read except `history` folds: it turns the rows
at or before its revision into the state they mean. How a
predicate folds is its cardinality.

### The ledger folds

The same `game/G1` rows, read at the top. Notice one `court`,
`south`: on a `one` predicate `add` is assignment and the latest
add wins. No retraction row was ever written for `north`.

In [ ]:
print(sorted(rev2.all("game/G1")))      # (predicate, object) pairs
# [('away', 'G5'), ('court', 'south'), ('date', '2026-10-11'),
#  ('home', 'G4'), ('officials', 'ortiz')]
print(rev2.get("game/G1", "court"))     # south
print(rev1.get("game/G1", "court"))     # north: rev1 never changes

`rev1` is a frozen view: a later transaction adds rows after it,
never changes what it reads.

### Without a declaration every predicate is one

`officials` was asserted twice, `kim` and `ortiz`, and the fold
kept only `ortiz`. Notice that this is wrong for officials: a game
has several. Nothing declared `officials`, so it folds as `one`,
the default, and the later add won.

In [ ]:
print(rev2.get("game/G1", "officials"))   # ortiz
print(rev2.all("game/G1", "officials"))   # ['ortiz']

### Declaring many, and the fold is as-of

A declaration is itself a row, on the predicate's own subject,
`db/pred/officials`, written by an ordinary transaction.

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/officials", "db/cardinality", "many")
rev3 = t.rev
print(rev3.all("game/G1", "officials"))   # ['kim', 'ortiz']
print(rev2.all("game/G1", "officials"))   # ['ortiz']
print(rev2.get("game/G1", "officials"))   # ortiz

Notice `kim` is back without being written again: from `rev3` on
every earlier `officials` add is a set member. The declaration is
as-of, so a read at `rev2` still folds `officials` as `one` and
still answers `ortiz`; reads of the past do not change.

### Accumulation

On a `many` predicate the members are a set. Notice `kim`, added
again while held, changes nothing.

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("game/G1", "officials", "lee")
    t.add("game/G1", "officials", "kim")   # held: no change
rev4 = t.rev
print(rev4.all("game/G1", "officials"))   # ['kim', 'ortiz', 'lee']

A remove, then a re-add. Notice `kim` comes back at the end.

In [ ]:
with L.transaction(by="human:dan") as t:
    t.remove("game/G1", "officials", "kim")
rev5 = t.rev
with L.transaction(by="human:dan") as t:
    t.add("game/G1", "officials", "kim")
rev6 = t.rev
print(rev5.all("game/G1", "officials"))   # ['ortiz', 'lee']
print(rev6.all("game/G1", "officials"))   # ['ortiz', 'lee', 'kim']

Members come in assertion order, each at the revision of its
earliest add not since removed: re-adding a present member (`kim`
at `rev4`) keeps its place; remove then re-add moves `kim` to the
end, at `rev6`.

### Flipping back

Declaring `one` again: from that revision on the latest add wins.
Notice the answer is `kim`, added last at `rev6`.

In [ ]:
with L.transaction(by="human:dan", why="one lead official") as t:
    t.add("db/pred/officials", "db/cardinality", "one")
rev7 = t.rev
print(rev7.get("game/G1", "officials"))   # kim: latest add wins

`rev6` is frozen at a revision where `officials` is `many`.
Notice `get` still refuses it there and names `all`.

In [ ]:
try:
    rev6.get("game/G1", "officials")
except TypeError as e:
    print("TypeError:", e)                 # many: use all

A flip writes and removes no `officials` row, so flipping to
`many` again brings the set back whole. Read at every revision:

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/officials", "db/cardinality", "many")
rev8 = t.rev
for n in range(rev1.id, rev8.id + 1):
    print(n, L.at(n).all("game/G1", "officials"))
# 1 []
# 2 ['ortiz']
# 3 ['kim', 'ortiz']
# 4 ['kim', 'ortiz', 'lee']
# 5 ['ortiz', 'lee']
# 6 ['ortiz', 'lee', 'kim']
# 7 ['kim']
# 8 ['ortiz', 'lee', 'kim']

Each `L.at(n)` folds every row at or before `n` by the
declarations at or before `n`; a revision is a view, not a copy.

## The `[]` accessor

A second game first, so the reads have two subjects.

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("game/G2", "home", "G1")
    t.add("game/G2", "away", "G3")
    t.add("game/G2", "date", "2026-10-12")
    t.add("game/G2", "court", "north")
rev9 = t.rev

`[]` reads the fold and returns the positions the pattern leaves
open. Three positions first: rows have revisions, so a held
triple answers with the revision of its earliest add not since
removed, on a `many` predicate too.

In [ ]:
rev = L.at()
print(rev["game/G1", "home", "G4"])           # Rev(1)
print(rev["game/G1", "officials", "kim"])     # Rev(6)
print(rev["game/G1", "officials", "ortiz"])   # Rev(2)

Notice a superseded or retracted triple raises rather than
answering with history.

In [ ]:
try:
    rev["game/G1", "court", "north"]
except KeyError as e:
    print("KeyError:", e)            # superseded at rev2: not held

Bracket reads take exact subjects: a glob is a `TypeError`, never
an empty answer; globs live on `all` and `select`.

In [ ]:
try:
    rev["game/*"]
except TypeError as e:
    print("TypeError:", e)

Two positions: the one object of a `one` predicate.

In [ ]:
print(rev["game/G1", "home"])        # G4
print(rev["game/G1", "court"])       # south: the latest add

Nothing there is a `KeyError`. Notice `game/G1` has no `score`
yet.

In [ ]:
try:
    rev["game/G1", "score"]
except KeyError as e:
    print("KeyError:", e)

A `many` predicate is refused by its declaration, however many
members it holds. Notice the message names `all`.

In [ ]:
try:
    rev["game/G1", "officials"]
except TypeError as e:
    print("TypeError:", e)                 # many: use all

One position: the record, a `dict`, which is the fold. A `one`
predicate maps to its object, a `many` predicate to its list, and
`db/` rows are in it like any other.

In [ ]:
print(rev["game/G1"])
# {'home': 'G4', 'away': 'G5', 'date': '2026-10-11',
#  'court': 'south', 'officials': ['ortiz', 'lee', 'kim']}
print(rev["game/G2"])
# {'home': 'G1', 'away': 'G3', 'date': '2026-10-12',
#  'court': 'north'}
print(rev["db/pred/officials"])      # {'db/cardinality': 'many'}

### Writing through the bracket

On a transaction the bracket is the mirror. `t[s, p] = o` is an
`add`; `t[s] = {...}` is closed world over the user rows. The
record write and `del` read what the subject holds before they
write, so they take `lock=` (see Transaction locking below). A
third game, written as a record, then rewritten without its `date`.

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    t["game/G3"] = {"home": "G2", "away": "G5",
                    "date": "2026-10-13", "court": "east"}
with L.transaction(by="human:dan", lock=10) as t:
    t["game/G3"] = {"home": "G2", "away": "G5", "court": "east"}
rev11 = t.rev
print(rev11["game/G3"] == {"home": "G2", "away": "G5",
                           "court": "east"})   # True: no date

Notice `date`, absent from the dict, was retracted. `del t[s, p]`
retracts what the predicate holds.

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    t["game/G3", "date"] = "2026-10-14"
    del t["game/G3", "court"]
rev12 = t.rev
print(rev12["game/G3", "date"])                # 2026-10-14
print(rev12.get("game/G3", "court"))           # None: retracted

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    t["game/G3"] = t["game/G3"]
noop_rev = t.rev
print([row for row in noop_rev.all(since=rev12)
       if not row[0].startswith("db/tx/")])   # []: nothing moved

Writing a record back as it was writes no data row; only the
transaction's own `db/tx/<rev>` rows are new. `db/` rows are
never touched by `t[s] = {...}`: a `db/` key equal to the current
row is ignored, which is what lets a record read with `db/` rows
be written back. Notice one that differs is refused: machinery is
not written through the bracket.

In [ ]:
try:
    with L.transaction(by="human:dan", lock=10) as t:
        t["game/G3"] = {**t["game/G3"], "db/doc": "a game"}
except TypeError as e:
    print("TypeError:", e)               # db/doc differs

`t.update(s, {...})` is an `add` per key of the record: it reads
nothing and retracts nothing, so it takes no `lock=`, and `db/`
keys go in as `add` takes them. Where `t[s] = {...}` is closed
world, `t.update` is open world: what the dict omits stays.

In [ ]:
with L.transaction(by="human:dan") as t:    # no lock=: nothing is read
    t.update("game/G3", {"court": "west"})
print("game/G3:", t.rev["game/G3"])
# → game/G3: {'home': 'G2', 'away': 'G5', 'date': '2026-10-14',
#    'court': 'west'}

Notice `home`, `away` and `date` stayed; `t["game/G3"] =
{"court": "west"}` would have retracted all three.

`t[s, p] = o` writes `one` predicates only; a record write
`t[s] = {...}` takes a list for a `many` key. Notice the refusal
names the predicate's cardinality.

In [ ]:
try:
    with L.transaction(by="human:dan") as t:
        t["game/G1", "officials"] = "park"
except TypeError as e:
    print("TypeError:", e)                 # t[s, p] = o: one only

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    del t["game/G3"]
print(t.rev.get("game/G3", "home"))      # None: every row retracted

`del t[s]` retracts the user rows: `game/G3` now holds nothing,
so `rev["game/G3"]` would raise `KeyError`.

## Triple accessors

Every read so far was one closed thing at one spot. These are the
ways to reach the ledger as triples: the softer two-position read,
the membership test, the history of a triple, every match of a
pattern, and subjects as a table.

### `get`, `in`, `history`

In [ ]:
rev = L.at()
print(rev.get("game/G1", "score"))                # None
print(rev.get("game/G1", "score", default="tbd")) # tbd
print(("game/G1", "home", "G4") in rev)           # True
print(("game/G1", "court", "north") in rev)       # False
for row in rev.history("game/G1", "court", "north"):  # its add; `south` replaced it, no remove row
    print(row)
#    1 + game/G1   court      north
print(rev.history("game/G1", "court", "west"))    # []
for row in rev.history("game/G1", "officials", "kim"):
    print(row)
#    2 + game/G1   officials  kim
#    4 + game/G1   officials  kim
#    5 - game/G1   officials  kim
#    6 + game/G1   officials  kim

`get` is `[]` answering `default` instead of raising on none.
`in` is the no-raise membership test, by the same fold.
`history` takes `all`'s pattern, unfolded: `north` is
gone from the fold and still in the history, and every add and
remove of `kim` is listed, in order.

### `all` binds the open positions

In [ ]:
print(sorted(rev.all("game/G1")))       # (predicate, object) pairs
# [('away', 'G5'), ('court', 'south'), ('date', '2026-10-11'),
#  ('home', 'G4'), ('officials', 'kim'), ('officials', 'lee'),
#  ('officials', 'ortiz')]
print(rev.all("game/G1", "home"))       # ['G4']: always a list
print(rev.all("game/G1", "home", "G4")) # [Rev(1)]
print([row for row in rev9.all(since=rev8)
       if not row[0].startswith("db/tx/")])
# game/G2's four rows: what (rev8, rev9] asserted
print(rev.get("game/G1", "court", since=rev1))  # south
print(rev.get("game/G1", "home", since=rev1))   # None: at rev1

`all` returns what the pattern leaves open, a list on either
cardinality, ordered by revision; rows of one revision have no
order, hence the `sorted`. `since=` restricts the rows considered
to an interval: `rev.all(since=a)` is the delta from `a`.

### Subjects as a table: `select`

In [ ]:
print(sorted(rev.select(("home", "away"), s="game/*")))
# [('G1', 'G3'), ('G4', 'G5')]
print(rev.select(("away",), home="G1"))       # [('G3',)]
print(rev.select(("court",), s="game/G2"))    # [('north',)]

A bare keyword means *has this row*, `s=` constrains the name
(exact or a prefix glob), and `select` projects predicates into
tuples. The spec gives `select` no order, so the notebook sorts.

In [ ]:
print(rev9.select(("home", "court"), where={"court": None},
                  since=rev8))               # [('G1', 'north')]

`since=` on the subject operations, in the Versions § 6 forms:
`where={"court": None}` is *has a court row, any value*, and only
`game/G2` asserted one in `(rev8, rev9]`; `game/G1`'s `south` is
at `rev2`.

`select` has siblings, `refs`, `values`, `value` and `load`, and
a `Ref` type: they hand back or instantiate values, and belong to
the values notebook. At this level nothing is ever instantiated:
every read here returns a `str` or a JSON document.

## Ledger management

Four topics that are about the ledger rather than about any one
subject: each is a row on a reserved subject or a keyword on a
transaction, nothing more.

### Declaring an index

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/home", "db/index", "true")
rev = L.at()
print(rev["db/pred/home"])                  # {'db/index': 'true'}
print(rev["db/pred/officials"])   # {'db/cardinality': 'many'}
print(("db/pred/home", "db/index", "true") in rev)   # True
print(rev.select(("away",), home="G1"))     # [('G3',)]

An index is storage: the declaration is a row on the predicate's
subject, beside `officials`' cardinality, and the store indexes
every existing `home` row. Notice the query answers the same as
before; an undeclared predicate is still a row, found by scan.

### Revision naming

A named revision is any subject's `rev` row, and `L.at` reads a
digit-only string as a revision id.

In [ ]:
with L.transaction(by="human:dan") as t:
    t["env/prod", "rev"] = str(rev2.id)
rev = L.at()
prod_rev = L.at(rev["env/prod", "rev"])       # L.at("2")
print(prod_rev.id, prod_rev.get("game/G1", "court"))   # 2 south

### Predicate typing: `db/type`

Checked at the write and on the strict reads.

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("game/G1", "score", {"home": 61, "away": 58})   # json
with L.transaction(by="human:dan") as t:
    t.add("db/pred/score", "db/type", "str")
    t.add("db/pred/date", "db/type", "str")
top_rev = L.at()

A predicate may declare the tag its objects take: `str`, `json`,
or a class's dotted name (the values notebook). Notice the
mismatching write is refused at commit and the top does not move.

In [ ]:
try:
    with L.transaction(by="human:dan") as t:
        t.add("game/G2", "date", {"y": 2026, "m": 10})
except TypeError as e:
    print("TypeError:", e)               # json under a str pred
print(L.at().id == top_rev.id)           # True: nothing written

Rows written before the declaration are not re-checked at the
write, but the strict read checks them. Notice `score`, stored as
JSON before it was declared `str`, raises here.

In [ ]:
rev = L.at()
try:
    rev["game/G1", "score"]
except TypeError as e:
    print("TypeError:", e)               # stored json, declared str

In [ ]:
print(rev.all("game/G1", "score"))       # [{'home': 61, ...}]
print(rev["game/G1"]["score"])           # the record: loose too
print(rev["game/G1", "date"])            # a str: passes

The loose reads, `all`, `rev[s]` and `select`, return what is
stored. A bare `int`, `float` or `bool` is refused by `add`
whatever the declaration: a number is stored as its string.

In [ ]:
try:
    with L.transaction(by="human:dan") as t:
        t.add("game/G1", "attendance", 1200)
except TypeError as e:
    print("TypeError:", e)               # a number is its string

In [ ]:
with L.transaction(by="human:dan") as t:
    t.add("game/G1", "attendance", "1200")

### A real code repository

Every value function runs in a git checkout, the ledger records
which commit, and the notebook makes a tiny one of its own; no
value function runs here, so one package declaring its
`__version__` is enough.

In [ ]:
import subprocess, sys

demo_repo_path = (project_path.parent / "demo-game").resolve()
(demo_repo_path / "demo_game").mkdir(parents=True, exist_ok=True)
(demo_repo_path / "demo_game/__init__.py").write_text('__version__ = "0.1.0"\n')
git_cmd = ["git", "-c", "user.name=demo",
           "-c", "user.email=demo@example.com"]
for args in (["init", "-q"], ["add", "."],
             ["commit", "-q", "-m", "demo-game 0.1.0"]):
    subprocess.run(git_cmd + args, cwd=demo_repo_path, check=True)
sys.path.insert(0, str(demo_repo_path))
import demo_game                       # discovery sees it imported

### The code clock

`db/repo/demo-game/demo_game` is a clock: its `db/version` is the
version of that package's code that runs were exposed to;
`db/repo/demo-game`'s `db/commit` is the commit of the latest clean
run. The ledger writes and reads both, so both carry `db/`. The first run from a clean
checkout writes both ([Ledger: dats](ledger-dats.ipynb) shows it);
this sheet runs nothing, so it adds the two rows such a run writes.

In [ ]:
demo_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=demo_repo_path, check=True,
                             capture_output=True, text=True).stdout.strip()
with L.transaction(by="human:dan") as t:
    t.add("db/repo/demo-game/demo_game", "db/version", demo_game.__version__)
    t.add("db/repo/demo-game", "db/commit", demo_commit)
print("db/repo/demo-game/demo_game:", L.at()["db/repo/demo-game/demo_game"])
# db/repo/demo-game/demo_game: {'db/version': '0.1.0'}
print("db/repo/demo-game:", L.at()["db/repo/demo-game"])
# db/repo/demo-game: {'db/commit': '<the demo commit>'}

### Transaction locking: `bump` under `lock=`

`bump` raises every other clock, here a schema's.

In [ ]:
with L.transaction(by="human:dan", lock=10,
                   why="T531 re-baseline") as t:
    new = bump(t, "schema/tape")          # patch: 0.0.0 -> 0.0.1
    t.add(t.tx, "checked", "G1,G4,G5")
rev = t.rev
print("schema/tape:", new)   # schema/tape: 0.0.1

`bump` reads the version through `t.previous_rev` and writes the
next one. `lock=10` holds the write lock for the body, so nothing
slips between the read and the write.

A body can do the same by hand. A transaction reads the previous
revision and writes the next; its own pending writes are not seen
until it closes. So the body reads with `t[s]`, alters, and writes
back. Here one more official joins `game/G1`.

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    g1 = t["game/G1"]                              # the record as it stands now
    t["game/G1"] = {**g1, "officials": g1["officials"] + ["vance"]}   # one more, written back
print("officials:", t.rev.all("game/G1", "officials"))
print("previous_rev:", repr(t.previous_rev))     # the revision the body read

`t.previous_rev` is the previous revision, with every read a
revision has: the top as it stands while the body holds the write
lock. The write is built on current data, and no other writer can
commit between the read and the write. Without `lock=` another
writer may commit after the body reads, so there is no
well-defined previous revision. Notice the ledger refuses the read.

In [ ]:
try:
    with L.transaction(by="human:dan") as t:       # no lock=
        t["game/G1"]
except Unlocked as e:
    print("Unlocked:", e)

`bump`, `t[s] = {...}` and `del` read through `t.previous_rev`, so
unlocked they refuse the same way and write nothing. `t[s, p] = o`
and `t.add` read nothing, so they need no lock.

In [ ]:
with L.transaction(by="human:dan") as t:           # no lock=: nothing is read
    t["game/G1", "court"] = "south"                # an add on a one predicate
    t.add("game/G1", "officials", "wu")            # an add on a many predicate
print("court:", t.rev["game/G1", "court"])

In [ ]:
print(sorted(rev[rev.tx]))
# ['checked', 'db/by', 'db/time', 'why']
print(rev[rev.tx, "db/time"])            # UTC, on every revision

`t.tx` is a placeholder for `db/tx/<rev>`, rewritten at commit:
provenance decided inside the body lands beside `by`, `db/time`
and the keywords given up front.

A minor or major bump must be confirmed. Notice the bare call
refuses and writes nothing.

In [ ]:
try:
    with L.transaction(by="human:dan", lock=10) as t:
        bump(t, "schema/tape", "minor")
except ValueError as e:
    print("ValueError:", e)              # minor needs confirm=

A code version lives in its source, so `bump` refuses a
`db/repo/` subject. Notice the message says where to bump it.

In [ ]:
try:
    with L.transaction(by="human:dan", lock=10) as t:
        bump(t, "db/repo/dvc-dat/dvc_dat")
except TypeError as e:
    print("TypeError:", e)               # db/repo/dvc-dat/dvc_dat: a code version lives in its source; …

In [ ]:
with L.transaction(by="human:dan", lock=10) as t:
    print(bump(t, "schema/tape", "minor",
               confirm=lambda old, new: True))   # 0.1.0
with L.transaction(by="human:dan") as t:
    t.add("schema/tape", "db/version", "0.0.9")
print(L.at()["schema/tape", "db/version"])  # 0.0.9: add went back

`confirm(old, new)` must answer `True` (`just bump` asks for the
word `yes`); a subject with no `db/version` starts from `0.0.0`.
`add` can set a clock backwards; `bump` is the helper that only
goes up.

In [ ]:
with L.transaction(by="agent:release", lock=5) as t:
    v = t["db/repo/demo-game/demo_game", "db/version"]
    c = t["db/repo/demo-game", "db/commit"]
    t.add("rel/demo-game", "pair", f"{v}+{c}")
print(L.at()["rel/demo-game", "pair"])    # 0.1.0+<commit>

Any read in the body holds still for the whole body. Notice that past the limit the body is interrupted and
`pair` is unchanged.

In [ ]:
try:
    with L.transaction(by="agent:release", lock=0.5) as t:
        t.add("rel/demo-game", "pair", "never")
        time.sleep(2)                     # past the limit
except LockTimeout as e:
    print("LockTimeout:", e)
print(L.at()["rel/demo-game", "pair"])    # 0.1.0+<commit>

In [ ]:
rev = L.at()
when = [r.rev for r in rev.history("db/repo/demo-game/demo_game", "db/version", "0.1.0") if r.added]
print(when)                               # [Rev(...)]: when it was set
print(when[0][when[0].tx, "db/by"])       # human:dan: by whom

The reverse lookup for a version: `history` names the revision,
and its `db/tx/<rev>` rows say who, and when if stamped.